In [1]:
from __future__ import print_function, unicode_literals, absolute_import, division
import sys, os
import numpy as np
#import pandas as pd

import matplotlib
matplotlib.rcParams["image.interpolation"] = 'none'
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from scipy import ndimage
import tifffile
import pandas as pd

from glob import glob
from tifffile import imread
from csbdeep.utils import Path, normalize
from csbdeep.io import save_tiff_imagej_compatible

from stardist import random_label_cmap, _draw_polygons, export_imagej_rois
from stardist.models import StarDist2D
from tqdm import tqdm

np.random.seed(6)
lbl_cmap = random_label_cmap()

2024-10-14 18:43:10.439705: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-10-14 18:43:12.257116: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-10-14 18:43:12.265083: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-14 18:43:24.489215: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)


In [3]:
def process_image(image_path, model):
    # Load image and ground truth coordinates
    image = tifffile.imread(image_path)
    h,w = image.shape


    axis_norm = (0,1)
    x = normalize(image,1,99.8,axis=axis_norm)
    mask = model.predict_instances(x, n_tiles=model._guess_n_tiles(x), show_tile_progress=False)[0]
             
    

    # Calculate the center coordinates from the predicted mask
    result_cofocal = ndimage.center_of_mass(mask, mask, np.unique(mask))
    resultList_cofocal = list(result_cofocal) 
    df_cofocal = pd.DataFrame(resultList_cofocal, columns=['row', 'column'])
    
    # Calculate number of pixel in each mask (ie cell)
    unique_labels, counts = np.unique(mask, return_counts = True)
    pixel_counts = dict(zip(unique_labels, counts))
    df_cofocal['mask_label'] = df_cofocal.index
    df_cofocal.drop(0, inplace=True, axis=0)  # Remove background NA
    df_cofocal['pixel_count'] = df_cofocal['mask_label'].map(pixel_counts)
    
    # Calculate area in square micron for each mask
    microns_per_pixel = 0.5639772144372924  # microns per pixel from metadata
    #mm_per_pixel = microns_per_pixel / 1000
    df_cofocal['mean cell area'] = df_cofocal['pixel_count'] * (microns_per_pixel ** 2) # area of each cone cell"μm²"
    
    # rearrange columns
    df_result = df_cofocal[['mask_label', 'row', 'column', 'pixel_count', 'mean cell area']].reset_index(drop=True)
    
    return df_result, h, w

In [4]:
def process_all_images(base_folder, high_model, low_model):
    # Use glob to find and sort the image file
    image_paths = sorted(glob(os.path.join(base_folder, 'AOSLO', '*.tiff')))
    
    microns_per_pixel = 0.5639772144372924
    mm_per_pixel = microns_per_pixel / 1000

    # Initialize lists to store the lengths
    cell_density = []
    average_areas = []
    degree = []
    cell_count = []
  
    # Process each image and its corresponding ground truth coordinates
    for image_path in tqdm(image_paths):
        
        try:
            # Extract filename from the full path
            filename = os.path.basename(image_path)
            
            if 'confocal' in filename:
                # Split filename into parts using '_'     # Extract the relevant parts
                parts = filename.split('_')
                value1 = int(parts[4])
                value2 = int(parts[5])

                # Filter based on degrees
                if -4 < value1 < 4 and -4 < value2 < 4: # for low degree only
                    model = low_model
                
                if not (-4 < value1 < 4 and -4 < value2 < 4): # for high degree only
                    model = high_model
                    
                df_result, h, w = process_image(image_path, low_model)
                N = len(df_result) # number of cells in a single image
                area_mm2 =  h * w * (mm_per_pixel ** 2)
                density = N / area_mm2
                average_area = np.sum(df_result['mean cell area'])  / N

                cell_density.append(density)
                average_areas.append(average_area)
                degree.append((value1,value2))
                cell_count.append(N)

        except (ValueError, IndexError):
            continue
 

    return cell_count, cell_density, average_areas, degree

In [5]:
# stardist
# high_model = StarDist2D(None, name='stardist_confocal_high', basedir='/home/hpc/iwi5/iwi5171h/stardist_src/stardist/examples/2D/models') # high 

# low_model = StarDist2D(None, name='stardist_confocal_low', basedir='/home/hpc/iwi5/iwi5171h/stardist_src/stardist/examples/2D/models') # low degree

high_model = StarDist2D(None, name='stardist_confocal_high', basedir='/home/hpc/iwi5/iwi5171h/thesis_src/trained_models/stardist_models') # high 

low_model = StarDist2D(None, name='stardist_confocal_low', basedir='/home/hpc/iwi5/iwi5171h/thesis_src/trained_models/stardist_models') # low degree

2024-10-14 18:43:44.796922: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.548138, nms_thresh=0.3.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.728766, nms_thresh=0.3.


In [6]:
AOSLO_list = [
    '4078_right_retina_Confocal_RPE_2023-09-09',
    '5181_left_retina_Confocal_RPE_IS_2023-09-10',
    '8323_right_retina_confocal_RPE_2023-09-13',
    '2538_right_retina_Confocal_RPE_2023-09-10',
    '4017_left_retina_Confocal_RPE_2023-09-10',
    '4064_od_retina_montaged_2023',
    '4571_left_retina_Confocal_RPE_IS_2023_09_09',
    '5160_right_retina_2024',
    '5160_right_retina_Confocal_RPE_2023-09-10',
    '5163_right_retina_RPE_2023-06-08',
    '5163_right_retina_2024',
    '5165_right_retina_RPE_2023-06-08',
    '5188_right_retina_Confocal_RPE_2023-09-10',
    '5225_right_retina_2024',
    '5232_right_retina_2024',
    '5213_right_retina_2024_session198',
    '5214_right_retina_2024_session205'
]

output_dir = 'analysis/csv_files'
if not os.path.exists(output_dir):
        os.makedirs(output_dir)

for AOSLO in tqdm(AOSLO_list):
    #base_folder = f'/home/hpc/iwi5/iwi5171h/data_processing/original_data/{AOSLO}/'
    base_folder = f'/home/hpc/iwi5/iwi5171h/thesis_src/multimodal_AOSLO_data/{AOSLO}/'
    
    
    # Process each AOSLO
    cell_count, pred_density, average_areas, degree = process_all_images(base_folder, high_model, low_model)
    
    # Creating a DataFrame 
    data = pd.DataFrame({
        'Degree': degree,
        'Cell Count': cell_count,
        'Predicted Cell Density': np.log10(pred_density), #"cones/mm²"
        'Average cell Area': average_areas #"μm²"
    })

    data = data.sort_values(by='Degree')
    data.reset_index(drop=True, inplace=True)
    
    # Save data to a CSV file
    data.to_csv(f'{output_dir}/{AOSLO}.csv', index=False)
    print(f'Processed and saved: {AOSLO}.csv')

  6%|▌         | 1/17 [01:18<20:58, 78.69s/it]

Processed and saved: 4078_right_retina_Confocal_RPE_2023-09-09.csv



 12%|█▏        | 2/17 [04:13<33:45, 135.03s/it]

Processed and saved: 5181_left_retina_Confocal_RPE_IS_2023-09-10.csv



 18%|█▊        | 3/17 [05:16<23:50, 102.21s/it]

...

Processed and saved: 5232_right_retina_2024.csv



 94%|█████████▍| 16/17 [31:29<02:20, 140.74s/it]

Processed and saved: 5213_right_retina_2024_session198.csv



100%|██████████| 17/17 [33:50<00:00, 119.44s/it]

Processed and saved: 5214_right_retina_2024_session205.csv
